# DiaTrace.AI: Clinical Entity Extraction & Fine-Tuning Pipeline
### Phase 2: Core NLP Fine-Tuning using Unsloth (QLoRA) on Llama-3-8B

Welcome to the **DiaTrace.AI Core NLP Engine** fine-tuning notebook. This interactive pipeline is designed to train a state-of-the-art Medical Entity Extraction model using memory-efficient QLoRA via **Unsloth**. The engine parses narrative clinical progress notes and extracts structured patient telemetry, biomarkers, and complex medication change logs into a **strict JSON format**.

#### Google Colab & Hugging Face Optimization:
1. **Direct Dataset Ingestion:** Configured to load the pre-generated `instruction_dataset.jsonl` directly (with automatic paths for Google Colab environment or local workspace).
2. **Hugging Face Hub Integration:** Features interactive Hugging Face login (`notebook_login`) and native Unsloth push methods to save either the lightweight LoRA adapters (~50MB) or the fully merged model (16-bit or 4-bit) directly to your Hugging Face account for production use at any time.
3. **Unsloth Speedups:** Utilizes Unsloth's highly optimized kernels to accelerate Llama-3-8B fine-tuning by up to 2x while slashing VRAM usage by 60% on free Colab GPU instances (T4, V100, or A100).

In [11]:
# Install Unsloth and matching xformers/pytorch/bitsandbytes
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "assets/sycl_kernel?" || pip install xformers

# Install helper libraries for supervised fine-tuning (TRL), training (PEFT, Accelerate), and evaluation
!pip install trl peft accelerate bitsandbytes datasets pydantic ipywidgets huggingface_hub
!pip install standard-imghdr || true
!pip install unsloth_zoo

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-fvi3r6z4/unsloth_b7ab6f7c2f1c441aa9a2f96e0a1549fe
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-fvi3r6z4/unsloth_b7ab6f7c2f1c441aa9a2f96e0a1549fe
  Resolved https://github.com/unslothai/unsloth.git to commit 56e9046b2ffd9d6e54d9cd25d4f37d7f24f7ed4e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: Invalid requirement: 'assets/sycl_kernel?': Expected end or semicolon (after name and no valid version specifier)
    assets/sycl_kernel?
          ^
Hint: It looks like a path. File 'assets/sycl_kernel?' does not exist.


In [15]:
import os
import json
import re
from datasets import Dataset

def load_instruction_dataset():
    """
    Scans active directories to locate 'instruction_dataset.jsonl' and loads it.
    Cleans invalid control characters that often break JSON parsing in clinical notes.
    """
    dataset_path = "instruction_dataset.jsonl" if os.path.exists("instruction_dataset.jsonl") else "/content/instruction_dataset.jsonl"

    if not os.path.exists(dataset_path):
        raise FileNotFoundError(f"Could not find {dataset_path}. Please upload the file.")

    print(f"Found dataset at: {os.path.abspath(dataset_path)}")

    records = []
    # Regex to catch control characters except for standard whitespace
    control_chars = re.compile(r'[\x00-\x1F\x7F-\x9F]')

    with open(dataset_path, 'r', encoding='utf-8') as f:
        for line_idx, line in enumerate(f):
            line = line.strip()
            if not line: continue

            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                try:
                    # Clean control characters and try again
                    cleaned_line = control_chars.sub('', line)
                    records.append(json.loads(cleaned_line))
                except Exception as e:
                    if line_idx < 5: # Only print first few errors to avoid log spam
                        print(f"Skipping line {line_idx} due to persistent error: {e}")

    print(f"Successfully loaded {len(records)} instruction-tuning pairs.")
    return Dataset.from_list(records)

# Re-load the dataset with enhanced cleaning
raw_dataset = load_instruction_dataset()

Found dataset at: /content/instruction_dataset.jsonl
Successfully loaded 39810 instruction-tuning pairs.


## Strict JSON Schema & Pydantic Validation

In [16]:
from pydantic import BaseModel, Field, ValidationError
from typing import List, Union, Literal

# Define structured output schema matching our extraction goals
class Biomarkers(BaseModel):
    eGFR: Union[int, float, Literal["Not Tested"]]
    HbA1c: Union[str, Literal["Not Tested"]]
    Glucose: Union[str, Literal["Not Tested"]]

class Medication(BaseModel):
    name: str
    status: Literal["Steady", "Up", "Down", "No"]

class ClinicalExtractionSchema(BaseModel):
    biomarkers: Biomarkers
    medications: List[Medication]

def validate_record(record_str: str) -> bool:
    """
    Ensures that a clinical output string perfectly adheres to our validation schema.
    """
    try:
        parsed = json.loads(record_str)
        ClinicalExtractionSchema(**parsed)
        return True
    except (json.JSONDecodeError, ValidationError) as e:
        print(f"Validation Failed: {e}")
        return False

# Validate the first record from the loaded dataset
sample_record = raw_dataset[0]
print("Sample Output:", sample_record["output"])
print("Is Valid Output Schema?:", validate_record(sample_record["output"]))

Sample Output: {"biomarkers": {"eGFR": 84, "HbA1c": "Not Tested", "Glucose": "Not Tested"}, "medications": [{"name": "glipizide", "status": "Steady"}, {"name": "insulin", "status": "Steady"}]}
Is Valid Output Schema?: True


## Initialize Unsloth & Load Llama-3-8B

In [17]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Supports custom context sizes (Unsloth supports RoPE scaling automatically!)
dtype = None # None for auto detection. Float16 for Tesla T4/V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to drastically reduce memory usage

# Load Qwen 2.5 7B optimized model via Unsloth
# Note: Qwen2.5-7B is the 9B parameter class model used locally (e.g., via Ollama/qwen3.5:9b).
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Apply Parameter-Efficient Fine-Tuning (PEFT) using LoRA (QLoRA)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank dimension (higher rank captures more complexity, but increases memory)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, # Optimized to 0 for maximum speed
    bias = "none",    # Optimized to none for maximum speed
    use_gradient_checkpointing = "unsloth", # Saves 30% VRAM by checkpointing gradients
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.6: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/172 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/617 [00:00<?, ?B/s]

unsloth/Qwen2.5-7B-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.5.6 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [18]:
import random

# Alpaca formatting template
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

def format_prompts(examples):
    """
    Applies the Alpaca formatting template to map inputs and outputs to a continuous string
    followed by an EOS token to guide model generation boundaries.
    """
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for inst, inp, out in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(inst, inp, out) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts }

# Apply Alpaca mapping
formatted_dataset = raw_dataset.map(format_prompts, batched = True)

# Perform Train/Test Split (95% Train, 5% Test)
split_dataset = formatted_dataset.train_test_split(test_size=0.05, seed=42)
train_set = split_dataset["train"]
test_set = split_dataset["test"]
print(f"Dataset split complete. Training on: {len(train_set)} records | Validating on: {len(test_set)} records")

Map:   0%|          | 0/39810 [00:00<?, ? examples/s]

Dataset split complete. Training on: 37819 records | Validating on: 1991 records


## Execute Fine-Tuning Loop using TRL SFTTrainer

In [19]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_set,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Packing short sequences together speeds up training significantly
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Set to 60 for quick validation. Increase to 500+ (or 1 full epoch) for production quality
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

# Start training
print("Initiating training loop using Unsloth optimized kernels...")
trainer_stats = trainer.train()
print(f"Training complete! Elapsed Time: {trainer_stats.metrics['train_runtime']:.2f} seconds")

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/37819 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Initiating training loop using Unsloth optimized kernels...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 37,819 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
5,2.182506
10,1.515612
15,0.560738
20,0.285708
25,0.238157
30,0.204881
35,0.193352
40,0.193267
45,0.188406
50,0.183495


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


Training complete! Elapsed Time: 457.65 seconds


## Inference Validation & Schema Guardrail Verification

Now that our model has been fine-tuned, we run an inference test on a holdout clinical progress note to verify that it generates beautifully formatted, strict, and highly accurate medical telemetry JSON output.

In [20]:
# Set the FastLanguageModel to optimized inference mode
FastLanguageModel.for_inference(model)

# Pick a test progress note from our validation dataset
sample_test = test_set[0]
test_note = sample_test["input"]
expected_output = sample_test["output"]

print("=== RAW CLINICAL NOTE (INPUT) ===")
print(test_note)
print("---------------------------------")

# Format according to the Alpaca template
inputs = tokenizer(
[
    alpaca_prompt.format(
        "You are a medical entity extraction system. Given a clinical progress note, extract the patient's biomarkers and active medications into a strict JSON format.",
        test_note,
        "" # Let the model auto-complete
    )
], return_tensors = "pt").to("cuda")

# Run generation with short output padding
outputs = model.generate(**inputs, max_new_tokens = 512, use_cache = True)
decoded_output = tokenizer.batch_decode(outputs)

# Parse response out of the Alpaca prompt output
response_section = decoded_output[0].split("### Response:\n")[-1].replace(EOS_TOKEN, "").strip()

print("=== GENERATED MED-NER EXTRACTED JSON ===")
print(response_section)
print("----------------------------------------")

# Verify schema integrity utilizing our Pydantic guards
is_valid = validate_record(response_section)
print(f"Schema Verification Result: {'PASS' if is_valid else 'FAIL'}")
print("Expected Ground Truth Output:", expected_output)

Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== RAW CLINICAL NOTE (INPUT) ===
CLINICAL PROGRESS NOTE - VISIT 2 OF 4
Date: 2026-05-23 | Patient ID: 38169090 | Encounter: 263028258
Demographics: 70 to 80 y.o. Caucasian male.
---------------------------------------------------------
SUBJECTIVE:
Patient admitted for comprehensive metabolic review (stay duration: 2 days). Patient reports compliance with current medications except during transient illnesses. Denies acute microvascular symptoms, tingling, or visual changes today. 

OBJECTIVE:
Physical Exam: Lower extremity sensation normal, monofilament check intact. Telemetry: 30 lab panels reviewed. HbA1c was not checked during this window. Calculated renal clearance (eGFR) is 69 mL/min/1.73m2. 

ASSESSMENT & PLAN:
1. Type 2 Diabetes Mellitus - suboptimally controlled. Complicated by diabetic nephropathy. 
Active Medication Matrix: Insulin (Up). Action taken: Insulin dosage adjusted up. 
---------------------------------


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

=== GENERATED MED-NER EXTRACTED JSON ===
{"biomarkers": {"eGFR": 69, "HbA1c": "Not Tested", "Glucose": "Not Tested"}, "medications": [{"name": "insulin", "status": "Up"}]}
----------------------------------------
Schema Verification Result: PASS
Expected Ground Truth Output: {"biomarkers": {"eGFR": 69, "HbA1c": "Not Tested", "Glucose": "Not Tested"}, "medications": [{"name": "insulin", "status": "Up"}]}


In [23]:
from huggingface_hub import login

# Provide your Hugging Face API token with 'Write' permissions
HF_TOKEN = "your_hf_token_here"
login(token=HF_TOKEN)

In [24]:
# Set your target repository name on Hugging Face
hf_repo_name = "raj0120/diatrace-qwen2.5-lora"

print(f"Pushing LoRA adapters to Hugging Face Hub: {hf_repo_name}...")

# Push only the adapters and tokenizer, not the full merged model
model.push_to_hub(hf_repo_name, token = HF_TOKEN)
tokenizer.push_to_hub(hf_repo_name, token = HF_TOKEN)

print("Success! LoRA adapters and tokenizer uploaded.")
print(f"You can now load these adapters using: model = FastLanguageModel.get_peft_model(base_model, '{hf_repo_name}')")

Pushing LoRA adapters to Hugging Face Hub: raj0120/diatrace-qwen2.5-lora...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  94%|#########4|  152MB /  162MB            

No files have been modified since last commit. Skipping to prevent empty commit.


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpphq8wmc9/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

No files have been modified since last commit. Skipping to prevent empty commit.


Success! LoRA adapters and tokenizer uploaded.
You can now load these adapters using: model = FastLanguageModel.get_peft_model(base_model, 'raj0120/diatrace-qwen2.5-lora')


In [ ]:
# Option A: Push the merged 16bit model (Base Model + LoRA weights combined)
# model.push_to_hub_merged("your-username/diatrace-llama3-merged", tokenizer, save_method = "merged_16bit")

# Option B: Push the merged 4bit model (extremely efficient for CPU/Edge device deployment)
# model.push_to_hub_merged("your-username/diatrace-llama3-merged-4bit", tokenizer, save_method = "merged_4bit")

In [28]:
from unsloth import FastLanguageModel
import torch
import gc

# Perform a deep cleanup of GPU memory
def cleanup_gpu():
    if 'model' in locals(): del model
    if 'tokenizer' in locals(): del tokenizer
    if 'trainer' in locals(): del trainer
    if 'loaded_model' in locals(): del loaded_model
    if 'loaded_tokenizer' in locals(): del loaded_tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

cleanup_gpu()

# Set repository name
hf_repo_name = "raj0120/diatrace-qwen2.5-lora"

# Load the LoRA adapters with reduced memory utilization to prevent CPU offloading
loaded_model, loaded_tokenizer = FastLanguageModel.from_pretrained(
    model_name = hf_repo_name,
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = load_in_4bit,
    gpu_memory_utilization = 0.6, # Reduced from 0.8 to give more buffer on T4
)

# Set to optimized inference mode
FastLanguageModel.for_inference(loaded_model)

print(f"Successfully loaded adapters from {hf_repo_name}")

# Test inference using the first record from the test set
sample_test = test_set[0]
inputs_loaded = loaded_tokenizer(
[
    alpaca_prompt.format(
        "You are a medical entity extraction system. Given a clinical progress note, extract the patient's biomarkers and active medications into a strict JSON format.",
        sample_test["input"],
        ""
    )
], return_tensors = "pt").to("cuda")

outputs_loaded = loaded_model.generate(**inputs_loaded, max_new_tokens = 512)
response_section_loaded = loaded_tokenizer.batch_decode(outputs_loaded)[0].split("### Response:\n")[-1].replace(EOS_TOKEN, "").strip()

print("\n=== GENERATED JSON ===")
print(response_section_loaded)
print(f"\nSchema Verification: {'PASS' if validate_record(response_section_loaded) else 'FAIL'}")

==((====))==  Unsloth 2026.5.6: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

unsloth/Qwen2.5-7B-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


adapter_model.safetensors:   0%|          | 0.00/162M [00:00<?, ?B/s]

Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Successfully loaded adapters from raj0120/diatrace-qwen2.5-lora


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)



=== GENERATED JSON ===
{"biomarkers": {"eGFR": 69, "HbA1c": "Not Tested", "Glucose": "Not Tested"}, "medications": [{"name": "insulin", "status": "Up"}]}

Schema Verification: PASS
